# SPM 3405: Machine Learning — Lecture 6 notebook
## Support vector machines

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Find the **maximal margin** line between pure and watered milk, and its **support vectors**.
2. See how the cost parameter **$C$** changes the margin when the classes overlap.
3. Use **kernels** to classify shaft-orbit readings, where the boundary is a circle.

Reading: ISLP 9.1–9.5. Run each cell with **Shift + Enter**, from top to bottom.

**Note on $C$:** in scikit-learn, **small $C$ = wide margin** (violations are cheap). ISLP's $C$ works the other way round.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers as the slides

def milk(n, sep):
    y = rng.integers(0, 2, n)                                                    # 1 = water added
    dens = np.where(y == 1, 1.0265 - sep * 0.002, 1.0305) + rng.normal(0, 0.0012, n)   # density (g/mL)
    cond = np.where(y == 1, 4.4 - sep * 0.4, 5.0) + rng.normal(0, 0.25, n)             # conductivity (mS/cm)
    return np.c_[dens, cond], y

def draw(model, Z, y, title):
    g1, g2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
    F = model.decision_function(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
    plt.contour(g1, g2, F, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="k")
    plt.scatter(*Z[y == 0].T, s=12, label="pure"); plt.scatter(*Z[y == 1].T, s=12, label="water added")
    plt.scatter(*Z[model.support_].T, s=80, facecolors="none", edgecolors="k", label="support vectors")
    plt.xlabel("density (standardised)"); plt.ylabel("conductivity (standardised)"); plt.title(title); plt.legend(); plt.show()

## Part 1: Dataset A — the classes are clearly apart (30 samples)

A very large $C$ allows no violations: this is the maximal margin classifier.

In [ ]:
X1, y1 = milk(30, 1.0)
Z1 = StandardScaler().fit_transform(X1)
mm = SVC(kernel="linear", C=1e6).fit(Z1, y1)
w, b = mm.coef_[0], mm.intercept_[0]
print("training errors:", np.sum(mm.predict(Z1) != y1))
print("support vectors:", mm.n_support_.sum(), "of", len(y1))
print("street width (standardised units): 2/|w| =", round(2 / np.linalg.norm(w), 2))
draw(mm, Z1, y1, "maximal margin classifier")

**Question 1.** Delete one sample that is **not** a support vector and refit. Does the line change? Now delete a support vector. What happens?

## Part 2: Dataset B — the classes overlap (200 training, 2000 test samples)

In [ ]:
X2, y2 = milk(200, -1.0); X2t, y2t = milk(2000, -1.0)
sc = StandardScaler().fit(X2); Z2, Z2t = sc.transform(X2), sc.transform(X2t)
cv = KFold(10, shuffle=True, random_state=0)
for Cval in [0.01, 0.1, 1, 10, 100]:
    s = SVC(kernel="linear", C=Cval).fit(Z2, y2)
    cv_err = 1 - cross_val_score(SVC(kernel="linear", C=Cval), Z2, y2, cv=cv).mean()
    print(f"C = {Cval:6}: support vectors {s.n_support_.sum():3d}   street width {2 / np.linalg.norm(s.coef_[0]):.2f}   "
          f"CV error {cv_err:.3f}   test error {np.mean(s.predict(Z2t) != y2t):.3f}")
print("logistic regression test error:", np.mean(LogisticRegression().fit(Z2, y2).predict(Z2t) != y2t).round(3))
for Cval in [0.01, 100]:
    draw(SVC(kernel="linear", C=Cval).fit(Z2, y2), Z2, y2, f"C = {Cval}")

**Questions**
2. As $C$ grows, what happens to the number of support vectors and the street width? Explain in words.
3. Cross-validation here prefers $C = 0.01$, yet on the 2000 test samples $C = 0.01$ is the worst. With only 200 samples, how big are the differences between the CV errors compared with their noise? (Lecture 4.)

## Part 3: Shaft orbit — a circular boundary

Two proximity probes give the shaft position $(x, y)$ in µm. Healthy shafts stay near the centre; faults swing far out in any direction.

In [ ]:
def orbit(n):
    y = (rng.uniform(size=n) < 0.4).astype(int)            # 1 = fault
    r = np.abs(np.where(y == 1, rng.normal(45, 10, n), rng.normal(22, 6, n)))
    th = rng.uniform(0, 2 * np.pi, n)
    return np.c_[r * np.cos(th), r * np.sin(th)], y

X3, y3 = orbit(300); X3t, y3t = orbit(3000)
sc3 = StandardScaler().fit(X3); Z3, Z3t = sc3.transform(X3), sc3.transform(X3t)

models = {
    "logistic regression": LogisticRegression().fit(Z3, y3),
    "linear SVM": GridSearchCV(SVC(kernel="linear"), {"C": [0.01, 0.1, 1, 10]}, cv=cv).fit(Z3, y3),
    "polynomial SVM (d=2)": GridSearchCV(SVC(kernel="poly", degree=2, coef0=1), {"C": [0.01, 0.1, 1, 10, 100]}, cv=cv).fit(Z3, y3),
    "radial SVM": GridSearchCV(SVC(kernel="rbf"), {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 0.5, 1, 5]}, cv=cv).fit(Z3, y3),
}
for name, m in models.items():
    best = getattr(m, "best_params_", "")
    print(f"{name:22s} test error {np.mean(m.predict(Z3t) != y3t):.3f}  {best}")

r = np.hypot(*X3t.T)
thr = np.linspace(20, 45, 501)
errs = [np.mean((r > t) != y3t) for t in thr]
print(f"best possible radius rule: fault if r > {thr[np.argmin(errs)]:.1f} µm, error {min(errs):.3f}")

In [ ]:
rbf = models["radial SVM"].best_estimator_
print("radial SVM support vectors:", rbf.n_support_.sum(), "of", len(y3))
g = np.linspace(-80, 80, 300); GX, GY = np.meshgrid(g, g)
F = rbf.decision_function(sc3.transform(np.c_[GX.ravel(), GY.ravel()])).reshape(GX.shape)
plt.figure(figsize=(5, 5))
plt.scatter(*X3[y3 == 0].T, s=8, label="healthy"); plt.scatter(*X3[y3 == 1].T, s=8, label="fault")
plt.contour(GX, GY, F, levels=[0], colors="k")
plt.xlabel("x (µm)"); plt.ylabel("y (µm)"); plt.legend(); plt.axis("equal"); plt.show()

**Questions**
4. Why do logistic regression and the linear SVM fail completely here?
5. Add the radius $r = \sqrt{x^2 + y^2}$ as a single input and fit logistic regression on it. What test error do you get? What does this say about knowing the physics?
6. Try the radial SVM with `gamma=5` and `C=100`. Draw the boundary. What has gone wrong?
7. **Your group project:** if your output is a category, add a radial SVM (tuned by CV) to your comparison table.

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `X2`, `y2`, `X2t`, `y2t`, `X3`, `y3`, `X3t`, `y3t`, `Z3`, `Z3t`, `sc3` and `cv` from above, so run the whole notebook first.

### Task 1: What happens if you forget to standardise?

An SVM measures distances, so the units of the inputs matter. Density varies by about 0.001 g/mL; conductivity by about 0.3 mS/cm.

(a) Fit a linear SVM (`C=1`) and a radial SVM (`C=1`, default `gamma`) to the milk dataset B **without** standardising (`X2`, `y2`),
and give their test errors on `X2t`, `y2t`.

(b) Repeat with standardised inputs (fit the scaler on the training data only). Compare.

(c) Change the units of density from g/mL to kg/m³ (multiply by 1000) and repeat (a). What changes, and why?
Which input does the unstandardised SVM effectively ignore?

In [ ]:
# Task 1: your code here

### Task 2: A map of $C$ and $\gamma$

For the radial SVM on the shaft-orbit data, the two tuning parameters work together.

(a) For $C$ in `[0.01, 0.1, 1, 10, 100, 1000]` and $\gamma$ in `[0.01, 0.1, 0.5, 1, 5, 20]`, compute the 10-fold CV error on `Z3`, `y3`.
Show the results as a heat map (`plt.imshow`) with the values written in the cells.

(b) Where on the map does the SVM **underfit**, and where does it **overfit**? Draw the boundary (as in Part 3) for one setting from each
region and for the best setting.

(c) How does the number of support vectors change with $\gamma$ at $C = 1$? Explain.

In [ ]:
# Task 2: your code here

### Task 3: SVM scores and a safe threshold

An SVM gives a **score** (`decision_function`), not a probability: positive means "fault", and larger means further from the boundary.
A plant engineer wants to catch **at least 95%** of shaft faults.

(a) Split the 300 training shafts into 200 for fitting and 100 for validation. Fit the radial SVM with the best $C$ and $\gamma$ from
Task 2 on the 200.

(b) On the validation shafts, find the largest score threshold that still catches at least 95% of the faults.

(c) Apply that threshold to the 3000 test shafts. What fraction of faults is caught, and what fraction of healthy shafts raise a
false alarm? Plot the ROC curve of the test scores and give the AUC.

In [ ]:
# Task 3: your code here